In [1]:
import os
import pandas as pd
import xarray as xr
import numpy as np

In [2]:
# Make sure logged in
import copernicusmarine
copernicusmarine.login()

INFO - 2026-03-18T17:55:09Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:

  eholmes


Copernicus Marine password:

  ········


File /home/jovyan/.copernicusmarine/.copernicusmarine-credentials already exists, overwrite it ? [y/N]:

  y


INFO - 2026-03-18T17:55:49Z - Credentials file stored in /home/jovyan/.copernicusmarine/.copernicusmarine-credentials.


True

In [2]:
def clean_dataset(dataset):
    """
    Rename dimensions with standard names and sort values if appropriate
    """
    # Rename dimensions if incorrect
    for coordinate in dataset.coords:
       if coordinate=='lon':
           dataset = dataset.rename({'lon': 'longitude'})
       if coordinate=='lat':
           dataset = dataset.rename({'lat': 'latitude'})
       if coordinate=='DEPTH':
           dataset = dataset.rename({'lat': 'depth'})
          
    # Sort correctly lon/lat if they were inverted
    for dimension in ["latitude","longitude"]:
        coordinates = dataset[dimension].values
        if (coordinates[0] >= coordinates[:-1]).all():
            dataset = dataset.sortby(dimension, ascending=True)
        
    return dataset

In [3]:
import pandas as pd
url = (
    "https://raw.githubusercontent.com/"
    "fish-pace/point-collocation/main/"
    "examples/fixtures/points.csv"
)
df_points = pd.read_csv(url)
print(len(df_points))

# Let's add on our own pc_id column
df_points = df_points.reset_index(drop=True)
df_points["pc_id"] = df_points.index + 1
df_points["pc_label"] = "pace_" + df_points["pc_id"].astype(str)
#df_points = clean_dataset(df_points)
df_points.head()

595


,lat,lon,date,pc_id,pc_label
0,27.3835,-82.7375,2024-06-13,1,pace_1
1,27.1190,-82.7125,2024-06-14,2,pace_2
2,26.9435,-82.8170,2024-06-14,3,pace_3
3,26.6875,-82.8065,2024-06-14,4,pace_4
4,26.6675,-82.6455,2024-06-14,5,pace_5


In [5]:
import pandas as pd
import numpy as np

# number of points
n = 100

# create dataframe
input_dataframe = pd.DataFrame({
    "LATITUDE": np.random.uniform(40, 42, n),
    "LONGITUDE": np.random.uniform(4, 8, n),
    "START": pd.to_datetime("2020-01-01"),
    "END": pd.to_datetime("2020-12-31"),
    "DEPTH": np.random.uniform(0, 10, n),
    "TIME": pd.to_datetime("2020-01-01")
})

print(input_dataframe.head())

    LATITUDE  LONGITUDE      START        END     DEPTH       TIME
0  41.135963   7.265483 2020-01-01 2020-12-31  9.450799 2020-01-01
1  40.845425   7.273647 2020-01-01 2020-12-31  3.178264 2020-01-01
2  40.203304   5.653118 2020-01-01 2020-12-31  2.947661 2020-01-01
3  41.580341   6.816275 2020-01-01 2020-12-31  4.303942 2020-01-01
4  40.665597   5.893063 2020-01-01 2020-12-31  8.854024 2020-01-01


In [6]:
# List of Copernicus Marine datasets (datasetIDs)
list_datasets = [
    'cmems_mod_med_phy-cur_my_4.2km_P1D-m',
]

# Output result names 
output_names = [
    'current_006_004',
]

# Variables to download
variables = ['uo','vo']

# SINGLE DATES ONLY: Set to 'True' if you want to keep depth/lat/lon/time from dataset (for checking purpose)
include_dataset_columns = True

# TIMESERIES ONLY: Set to 'True' to save output files in NetCDF format instead of CSV
save_as_netcdf = False


In [7]:
for dataset_id, output_name in zip(list_datasets, output_names):

    # Read dataset via Toolbox
    dataset = copernicusmarine.open_dataset(dataset_id=dataset_id, chunk_size_limit=0) # chunk size limit set to 0 for single time steps

    # Clean dataset
    dataset = clean_dataset(dataset)

    # Select only requested variables first (important for correct depth detection)
    selected_variables_dataset = dataset[variables]

    # Copy input dataframe
    input_points_dataframe = input_dataframe.copy()

    # Build vectorized indexers
    time_points = xr.DataArray(pd.to_datetime(input_points_dataframe["TIME"]).to_numpy(), dims="points")
    latitude_points = xr.DataArray(input_points_dataframe["LATITUDE"].to_numpy(dtype=float), dims="points")
    longitude_points = xr.DataArray(input_points_dataframe["LONGITUDE"].to_numpy(dtype=float), dims="points")

    selection_indexers = {
        "time": time_points,
        "latitude": latitude_points,
        "longitude": longitude_points,
    }

    # Depth detection must be done on the selected variables dataset (not the full dataset)
    dataset_has_depth = "depth" in selected_variables_dataset.dims
    dataframe_has_depth = "DEPTH" in input_points_dataframe.columns and input_points_dataframe["DEPTH"].notna().any()

    # Download data for 3D variables
    if dataset_has_depth:
        if dataframe_has_depth:
            depth_points = xr.DataArray(input_points_dataframe["DEPTH"].to_numpy(dtype=float), dims="points")
            selection_indexers["depth"] = depth_points
            selected_points_dataset = selected_variables_dataset.sel(**selection_indexers, method="nearest")
        else:
            selected_points_dataset = selected_variables_dataset.isel(depth=0).sel(**selection_indexers, method="nearest")
    else:
        # Download data for 2D variables
        selected_points_dataset = selected_variables_dataset.sel(**selection_indexers, method="nearest")

    # Convert dataset to dataframe
    downloaded_variables_dataframe = selected_points_dataset.to_dataframe().reset_index(drop=True)

    # Identify index-like columns coming from xarray
    index_like_columns = [column for column in ("time", "latitude", "longitude", "depth") if column in downloaded_variables_dataframe.columns]

    # Keep depth/lat/lon/time from the dataset only if requested
    if not include_dataset_columns:
        downloaded_variables_dataframe = downloaded_variables_dataframe.drop(columns=index_like_columns, errors="ignore")

    # Add variables to input dataframe
    output_dataframe = pd.concat(
        [input_points_dataframe.reset_index(drop=True), downloaded_variables_dataframe.reset_index(drop=True)],
        axis=1,
    )

    output_dataframe

INFO - 2026-03-18T17:56:18Z - Selected dataset version: "202511"
INFO - 2026-03-18T17:56:18Z - Selected dataset part: "default"


In [8]:
output_dataframe

,LATITUDE,LONGITUDE,START,END,DEPTH,TIME,uo,vo,depth,latitude,longitude,time
0,41.836353,5.263667,2020-01-01,2020-12-31,6.575323,2020-01-01,-0.050474,-0.022055,5.464963,41.854168,5.250000,2020-01-01
1,41.320826,4.778977,2020-01-01,2020-12-31,3.133180,2020-01-01,0.296798,-0.036456,3.165747,41.312500,4.791667,2020-01-01
2,40.223916,7.108160,2020-01-01,2020-12-31,6.065819,2020-01-01,-0.035955,0.074415,5.464963,40.229168,7.125000,2020-01-01
3,40.837865,5.674847,2020-01-01,2020-12-31,2.860700,2020-01-01,0.126110,0.086549,3.165747,40.854168,5.666667,2020-01-01
4,40.396755,7.382790,2020-01-01,2020-12-31,4.942303,2020-01-01,0.131832,-0.137944,5.464963,40.395832,7.375000,2020-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...
95,41.262296,7.552853,2020-01-01,2020-12-31,0.313150,2020-01-01,-0.059806,0.139902,1.018237,41.270832,7.541667,2020-01-01
96,40.764744,4.223551,2020-01-01,2020-12-31,3.808210,2020-01-01,0.050801,0.055411,3.165747,40.770832,4.208333,2020-01-01
97,41.729007,7.744567,2020-01-01,2020-12-31,3.175710,2020-01-01,-0.024087,-0.275042,3.165747,41.729168,7.750000,2020-01-01
98,40.157205,4.462016,2020-01-01,2020-12-31,9.004257,2020-01-01,0.238996,-0.168046,7.920377,40.145832,4.458333,2020-01-01


In [11]:
dataset

<xarray.Dataset> Size: 6TB
Dimensions:    (depth: 141, latitude: 380, longitude: 1016, time: 14304)
Coordinates:
  * depth      (depth) float32 564B 1.018 3.166 5.465 ... 5.646e+03 5.754e+03
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -6.0 -5.958 -5.917 ... 36.21 36.25 36.29
  * time       (time) datetime64[ns] 114kB 1987-01-01 1987-01-02 ... 2026-02-28
Data variables:
    uo         (time, depth, latitude, longitude) float32 3TB ...
    vo         (time, depth, latitude, longitude) float32 3TB ...
Attributes:
    title:        Horizontal Velocity (3D) - Daily Mean
    source:       MFS E3R1I
    institution:  Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC, ...
    contact:      servicedesk.cmems@mercator-ocean.eu
    comment:      Please check in CMEMS catalogue the INFO section for produc...
    Conventions:  CF-1.0
    references:   Escudier, R., Clementi, E., Omar, M., Cipollone, A., Pistoi...

In [13]:
import numpy as np

lon_idx = np.linspace(0, dataset.sizes["longitude"] - 1, 10, dtype=int)
lat_idx = np.linspace(0, dataset.sizes["latitude"] - 1, 10, dtype=int)
depth_idx = np.linspace(0, dataset.sizes["depth"] - 1, 10, dtype=int)

sub = (
    dataset.sel(time=slice("2024-01-01", "2024-12-31"))
        .isel(depth=depth_idx)
        .isel(longitude=lon_idx, latitude=lat_idx)
)
time_idx = np.linspace(0, sub.sizes["time"] - 1, 20, dtype=int)
sub = sub.isel(time=time_idx)

In [14]:
sub.nbytes/1e6

0.16028

In [15]:
sub.to_netcdf("multi_time_depth.nc")

In [5]:
import copernicusmarine
dataset_id = 'cmems_mod_med_phy-cur_my_4.2km_P1D-m'
dataset = copernicusmarine.open_dataset(dataset_id=dataset_id, chunk_size_limit=0)

INFO - 2026-03-18T20:53:53Z - Selected dataset version: "202511"
INFO - 2026-03-18T20:53:53Z - Selected dataset part: "default"


In [6]:
dataset

<xarray.Dataset> Size: 6TB
Dimensions:    (depth: 141, latitude: 380, longitude: 1016, time: 14304)
Coordinates:
  * depth      (depth) float32 564B 1.018 3.166 5.465 ... 5.646e+03 5.754e+03
  * latitude   (latitude) float32 2kB 30.19 30.23 30.27 ... 45.9 45.94 45.98
  * longitude  (longitude) float32 4kB -6.0 -5.958 -5.917 ... 36.21 36.25 36.29
  * time       (time) datetime64[ns] 114kB 1987-01-01 1987-01-02 ... 2026-02-28
Data variables:
    uo         (time, depth, latitude, longitude) float32 3TB ...
    vo         (time, depth, latitude, longitude) float32 3TB ...
Attributes:
    references:   Escudier, R., Clementi, E., Omar, M., Cipollone, A., Pistoi...
    title:        Horizontal Velocity (3D) - Daily Mean
    contact:      servicedesk.cmems@mercator-ocean.eu
    Conventions:  CF-1.0
    comment:      Please check in CMEMS catalogue the INFO section for produc...
    institution:  Centro Euro-Mediterraneo sui Cambiamenti Climatici - CMCC, ...
    source:       MFS E3R1I